# Credit Card Approval Prediction

Automating credit card approval decisions using machine learning.

This notebook covers:
1. Loading & merging the raw applicant + credit history data
2. Exploratory Data Analysis (EDA)
3. Feature engineering (including converting multi-class payment status into a binary risk label)
4. Preprocessing (encoding + scaling)
5. Training 4 classifiers: Logistic Regression, Decision Tree, Random Forest, XGBoost
6. Comparing model performance
7. Saving the best model for the Flask web app


In [ ]:
import sys
sys.path.append('../src')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', 50)
%matplotlib inline


## 1. Load Raw Data

If you don't have the real Kaggle dataset, run `python ../data/generate_dataset.py` first to create a synthetic one with the same schema.

In [ ]:
application_df = pd.read_csv('../data/application_record.csv')
credit_df = pd.read_csv('../data/credit_record.csv')

print('Applications:', application_df.shape)
print('Credit history rows:', credit_df.shape)
application_df.head()


In [ ]:
credit_df.head()


## 2. Exploratory Data Analysis

In [ ]:
application_df.info()


In [ ]:
application_df.describe()


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(application_df['AMT_INCOME_TOTAL'], bins=50, ax=axes[0])
axes[0].set_title('Income Distribution')
sns.countplot(x='NAME_INCOME_TYPE', data=application_df, ax=axes[1])
axes[1].set_title('Income Type Counts')
axes[1].tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()


In [ ]:
status_counts = credit_df['STATUS'].value_counts()
status_counts.plot(kind='bar', figsize=(8,4), title='Payment Status Code Distribution')
plt.xlabel('STATUS code')
plt.ylabel('Count')
plt.show()
print(status_counts)


## 3. Feature Engineering

Key step: the multi-class `STATUS` payment codes (`0`-`5`, `C`, `X`) are converted into a **binary** high-risk flag.
`2`, `3`, `4`, `5` represent 60+ days past due. An applicant with 2 or more such months is
labelled **high risk -> Rejected (0)**; otherwise **Approved (1)**.

We also derive `AGE_YEARS`, `EMPLOYMENT_YEARS`, `IS_PENSIONER`, and `INCOME_PER_FAMILY_MEMBER`
from the raw fields.

In [ ]:
from feature_engineering import build_dataset, FEATURE_COLUMNS, TARGET_COLUMN

dataset = build_dataset('../data/application_record.csv', '../data/credit_record.csv')
print(dataset.shape)
dataset.head()


In [ ]:
dataset['TARGET'].value_counts(normalize=True).plot(
    kind='bar', title='Target Class Balance (1=Approved, 0=Rejected)'
)
plt.show()
dataset['TARGET'].value_counts()


In [ ]:
plt.figure(figsize=(6,4))
sns.boxplot(x='TARGET', y='AMT_INCOME_TOTAL', data=dataset)
plt.title('Income vs Approval Outcome')
plt.ylim(0, dataset['AMT_INCOME_TOTAL'].quantile(0.95))
plt.show()


## 4. Preprocessing & Train/Test Split

In [ ]:
from data_preprocessing import build_preprocessor, get_train_test_split

X_train, X_test, y_train, y_test = get_train_test_split(dataset)
print('Train:', X_train.shape, ' Test:', X_test.shape)


## 5. Train Four Classifiers

Logistic Regression, Decision Tree, Random Forest, and XGBoost (falls back to `GradientBoostingClassifier` automatically if `xgboost` isn't installed).

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, ConfusionMatrixDisplay, classification_report
)

try:
    from xgboost import XGBClassifier
    XGBOOST_AVAILABLE = True
except ImportError:
    XGBOOST_AVAILABLE = False

models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, class_weight='balanced'),
    'Decision Tree': DecisionTreeClassifier(max_depth=8, random_state=42, class_weight='balanced'),
    'Random Forest': RandomForestClassifier(n_estimators=300, max_depth=12, random_state=42, class_weight='balanced', n_jobs=-1),
}
if XGBOOST_AVAILABLE:
    models['XGBoost'] = XGBClassifier(n_estimators=300, max_depth=6, learning_rate=0.1, eval_metric='logloss', random_state=42)
else:
    print('xgboost not installed - using GradientBoostingClassifier as a stand-in')
    models['XGBoost (fallback)'] = GradientBoostingClassifier(n_estimators=300, max_depth=4, learning_rate=0.1, random_state=42)


In [ ]:
results = {}
fitted_pipelines = {}

for name, clf in models.items():
    pipe = Pipeline([('preprocessor', build_preprocessor()), ('classifier', clf)])
    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)
    y_proba = pipe.predict_proba(X_test)[:, 1]

    results[name] = {
        'accuracy': accuracy_score(y_test, y_pred),
        'precision': precision_score(y_test, y_pred),
        'recall': recall_score(y_test, y_pred),
        'f1_score': f1_score(y_test, y_pred),
        'roc_auc': roc_auc_score(y_test, y_proba),
    }
    fitted_pipelines[name] = pipe
    print(f'--- {name} ---')
    print(classification_report(y_test, y_pred, target_names=['Rejected', 'Approved']))


## 6. Compare Models

In [ ]:
results_df = pd.DataFrame(results).T.sort_values('f1_score', ascending=False)
results_df


In [ ]:
results_df.plot(kind='bar', figsize=(10, 6), rot=20, title='Model Comparison')
plt.ylabel('Score')
plt.tight_layout()
plt.show()


In [ ]:
best_name = results_df.index[0]
best_pipeline = fitted_pipelines[best_name]
print('Best model:', best_name)

y_pred_best = best_pipeline.predict(X_test)
cm = confusion_matrix(y_test, y_pred_best)
ConfusionMatrixDisplay(cm, display_labels=['Rejected', 'Approved']).plot()
plt.title(f'Confusion Matrix - {best_name}')
plt.show()


## 7. Save Best Model

The full pipeline (preprocessing + classifier) is saved so the Flask app can call `.predict()` directly on raw feature rows.

In [ ]:
import joblib, os

os.makedirs('../models', exist_ok=True)
joblib.dump(best_pipeline, '../models/best_model.pkl')

os.makedirs('../flask_app/model', exist_ok=True)
joblib.dump(best_pipeline, '../flask_app/model/best_model.pkl')

print('Saved best_model.pkl to ../models and ../flask_app/model')
